# QA Checks for Gold Views


### 1. Row Count & Null Checks (run after refresh)

In [0]:
%sql
-- Sanity check all three views
SELECT 'daily_metrics_1' AS view, COUNT(*) AS rows FROM workspace.gold.daily_metrics_1
UNION ALL
SELECT 'daily_metrics_2', COUNT(*) FROM workspace.gold.daily_metrics_2
UNION ALL
SELECT 'daily_metrics_3', COUNT(*) FROM workspace.gold.daily_metrics_3;




### 2. Verify totals match silver (the source of truth)

In [0]:
%sql
-- Gold total events should match silver total rows
SELECT 
  (SELECT SUM(total_events) FROM workspace.gold.daily_metrics_1) AS gold1_total,
  (SELECT SUM(total_events) FROM workspace.gold.daily_metrics_2) AS gold2_total,
  (SELECT SUM(total_events) FROM workspace.gold.daily_metrics_3) AS gold3_total,
  (SELECT COUNT(*) FROM workspace.silver.ga4_silver)             AS silver_total;
--All three gold totals should equal silver_total.

### 3. Check for nulls in group-by columns

In [0]:
%sql
-- Nulls in dimensions would cause silent data loss in groupings
SELECT COUNT(*) AS null_country   FROM workspace.gold.daily_metrics_1 WHERE country IS NULL;

SELECT COUNT(*) AS null_event     FROM workspace.gold.daily_metrics_2 WHERE event_name IS NULL;

SELECT COUNT(*) AS null_page      FROM workspace.gold.daily_metrics_3 WHERE page_location IS NULL;

### 4. Date range check — no gaps, no future dates

In [0]:
%sql
SELECT 
  MIN(event_date) AS earliest,
  MAX(event_date) AS latest,
  COUNT(DISTINCT event_date) AS distinct_days
FROM workspace.gold.daily_metrics_1;
--Cross-check this against what you expect from the silver layer.

### 5. Top-N spot check (eyeball the data)

### 5. Top-N spot check (eyeball the data)

In [0]:
%sql
SELECT * FROM workspace.gold.daily_metrics_1 ORDER BY total_events DESC LIMIT 10;
SELECT * FROM workspace.gold.daily_metrics_2 ORDER BY total_events DESC LIMIT 10;
SELECT * FROM workspace.gold.daily_metrics_3 ORDER BY total_events DESC LIMIT 10;
